In [6]:
import os

import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

In [7]:
device = 0 if torch.backends.mps.is_available() else -1

In [8]:
classifier = pipeline(
    "text-classification", 
    model="monologg/bert-base-cased-goemotions-original",
    top_k=None,
    device=device,
    token=os.getenv("HF_TOKEN")
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
def get_emotion_vector(text):
    clean_text = str(text)[:512]
    results = classifier(clean_text)
    return [res['score'] for res in results[0]]

In [10]:
df = pd.read_csv("../../../data/cleaned_data.csv")
features = []
for text in tqdm(df['cleaned_statement']):
    features.append(get_emotion_vector(text))

100%|██████████| 31993/31993 [05:15<00:00, 101.42it/s]


In [11]:
emotion_cols = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]
features_df = pd.DataFrame(features, columns=emotion_cols)
features_df['label'] = df['label']

In [12]:
features_df.to_csv('../../../data/features_for_model.csv', index=False)